# 🏆 FIFA World Cup Prediction — Notebook 3: Modelling

**What this notebook builds:**

| Model | Target | Method |
|---|---|---|
| Poisson match model | Home/away goals | Team attack & defense ratings |
| Tournament simulator | Winner probability | Monte Carlo (10,000 runs) |
| Award ranker — Golden Boot | Top scorer | Ridge regression on player features |
| Award ranker — Best Player | Best player | Ridge regression |
| Award ranker — Young Player | Best U21 player | Ridge regression filtered by age |
| Baseline comparison | All targets | Elo-only benchmark |

**Input files (from Notebook 2):**
- `team_features.csv`
- `player_features.csv`

These are loaded at the top. If not found, the notebook re-scrapes automatically.


## 1. Setup

In [ ]:
!pip install requests pandas numpy matplotlib seaborn scikit-learn scipy --quiet

import requests, io, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, mean_absolute_error
from collections import defaultdict

warnings.filterwarnings("ignore")
np.random.seed(42)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")
print("✅ Ready")

## 2. Load Data

Loads from saved CSV files if available. If not found, re-scrapes from GitHub.


In [ ]:
try:
    team_features   = pd.read_csv("team_features.csv")
    player_features = pd.read_csv("player_features.csv")
    print(f"✅ Loaded from CSV")
except FileNotFoundError:
    print("CSV files not found — re-scraping from GitHub...")
    # Paste and run Notebook 2 scrape cell here if needed, or upload the files
    raise FileNotFoundError("Run Notebook 2 first to generate team_features.csv and player_features.csv")

# Also fetch raw matches for the Poisson model (needs match-level data)
BASE = "https://raw.githubusercontent.com/jfjelstul/worldcup/master/data-csv/"
def fetch(f):
    r = requests.get(BASE + f, timeout=15); r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text), low_memory=False)

tournaments = fetch("tournaments.csv")
raw_matches  = fetch("matches.csv")

men_tourns = tournaments[tournaments["tournament_name"].str.contains("Men", na=False)]
YEAR_MAP   = men_tourns.set_index("tournament_id")["year"].to_dict()
WINNER_MAP = men_tourns.set_index("tournament_id")["winner"].to_dict()

def men_only(df):
    df = df.copy(); df["year"] = df["tournament_id"].map(YEAR_MAP)
    return df[df["year"].notna()].reset_index(drop=True)

matches = men_only(raw_matches)
matches["year"]            = matches["year"].astype(int)
matches["home_team_score"] = pd.to_numeric(matches["home_team_score"], errors="coerce")
matches["away_team_score"] = pd.to_numeric(matches["away_team_score"], errors="coerce")
matches["match_date"]      = pd.to_datetime(matches["match_date"], errors="coerce")
matches = matches.dropna(subset=["home_team_score","away_team_score"]).reset_index(drop=True)

team_features["year"]   = team_features["year"].astype(int)
player_features["year"] = player_features["year"].astype(int)

print(f"\nteam_features   : {team_features.shape}")
print(f"player_features : {player_features.shape}")
print(f"matches         : {matches.shape}")
print(f"Years covered   : {sorted(matches['year'].unique())[-5:]}")

## 3. Train / Test Split

**Split by year, not randomly.**

Random splitting would let the model train on 2018 data and test on 2014 — it would
already "know" which teams were good in 2018 when predicting 2014. That's data leakage.

We use the last two tournaments (2018, 2022) as test sets and everything before as training.


In [ ]:
TEST_YEARS  = [2018, 2022]
TRAIN_YEARS = [y for y in sorted(matches["year"].unique()) if y not in TEST_YEARS]

train_matches = matches[matches["year"].isin(TRAIN_YEARS)]
test_matches  = matches[matches["year"].isin(TEST_YEARS)]

train_teams   = team_features[team_features["year"].isin(TRAIN_YEARS)]
test_teams    = team_features[team_features["year"].isin(TEST_YEARS)]

train_players = player_features[player_features["year"].isin(TRAIN_YEARS)]
test_players  = player_features[player_features["year"].isin(TEST_YEARS)]

print(f"Train matches : {len(train_matches):,}  |  years: {TRAIN_YEARS[-3:]}...")
print(f"Test matches  : {len(test_matches):,}   |  years: {TEST_YEARS}")
print(f"Train teams   : {len(train_teams):,}")
print(f"Test teams    : {len(test_teams):,}")

# Check for unseen teams in test set
train_team_names = set(train_matches["home_team_name"]) | set(train_matches["away_team_name"])
test_team_names  = set(test_matches["home_team_name"])  | set(test_matches["away_team_name"])
unseen = test_team_names - train_team_names
print(f"\nUnseen teams in test set: {unseen}")
print("→ These get league-average attack/defense ratings (handled below)")

## 4. Poisson Match Model

Models goals scored as Poisson-distributed with:

```
home_goals ~ Poisson(attack[home] × defense[away] × home_advantage)
away_goals ~ Poisson(attack[away] × defense[home])
```

Team attack and defense ratings are estimated iteratively from historical match results.
Higher attack = scores more. Higher defense = concedes more (so lower = better defending).


In [ ]:
def fit_poisson_ratings(df, n_iter=10):
    """
    Iteratively estimate team attack and defense ratings.
    Attack > 1 = above-average scorer. Defense < 1 = above-average defender.
    """
    teams   = sorted(set(df["home_team_name"]) | set(df["away_team_name"]))
    attack  = {t: 1.0 for t in teams}
    defense = {t: 1.0 for t in teams}
    home_adv = df["home_team_score"].mean() / max(df["away_team_score"].mean(), 0.01)

    for _ in range(n_iter):
        for team in teams:
            h = df[df["home_team_name"] == team]
            a = df[df["away_team_name"] == team]

            actual_gf = list(h["home_team_score"]) + list(a["away_team_score"])
            actual_ga = list(h["away_team_score"]) + list(a["home_team_score"])

            exp_gf = (
                [attack[team] * defense[opp] * home_adv for opp in h["away_team_name"]] +
                [attack[team] * defense[opp]             for opp in a["home_team_name"]]
            )
            exp_ga = (
                [attack[opp] * defense[team]             for opp in h["away_team_name"]] +
                [attack[opp] * defense[team] * home_adv  for opp in a["home_team_name"]]
            )

            if sum(exp_gf) > 0: attack[team]  = sum(actual_gf) / sum(exp_gf)
            if sum(exp_ga) > 0: defense[team] = sum(actual_ga) / sum(exp_ga)

    return attack, defense, home_adv

attack, defense, home_adv = fit_poisson_ratings(train_matches)

print(f"Home advantage multiplier: {home_adv:.3f}")
print(f"\nTop 10 attack ratings:")
top_attack = sorted(attack.items(), key=lambda x: x[1], reverse=True)[:10]
for team, val in top_attack:
    print(f"  {team:<25} {val:.3f}")

print(f"\nTop 10 defense ratings (lower = better):")
top_def = sorted(defense.items(), key=lambda x: x[1])[:10]
for team, val in top_def:
    print(f"  {team:<25} {val:.3f}")

In [ ]:
# Predict match outcomes
AVG_ATTACK  = np.mean(list(attack.values()))
AVG_DEFENSE = np.mean(list(defense.values()))

def predict_match(home, away, attack, defense, home_adv):
    """
    Returns (P(home win), P(draw), P(away win)) and expected goals.
    Uses full Poisson score distribution, not just means.
    """
    lam_h = attack.get(home, AVG_ATTACK) * defense.get(away, AVG_DEFENSE) * home_adv
    lam_a = attack.get(away, AVG_ATTACK) * defense.get(home, AVG_DEFENSE)

    max_goals = 10
    p_home_win = p_draw = p_away_win = 0.0

    for hg in range(max_goals + 1):
        for ag in range(max_goals + 1):
            p = poisson.pmf(hg, lam_h) * poisson.pmf(ag, lam_a)
            if hg > ag: p_home_win += p
            elif hg == ag: p_draw   += p
            else:          p_away_win += p

    return p_home_win, p_draw, p_away_win, lam_h, lam_a

# Test on some 2022 matches
test_fixtures = [
    ("Argentina", "France"),
    ("Brazil", "Croatia"),
    ("England", "France"),
    ("Morocco", "Portugal"),
]
print(f"{'Match':<30} {'P(Home)':<10} {'P(Draw)':<10} {'P(Away)':<10} {'xG Home':<10} {'xG Away'}")
print("-"*80)
for home, away in test_fixtures:
    ph, pd_, pa, lh, la = predict_match(home, away, attack, defense, home_adv)
    print(f"{home+' vs '+away:<30} {ph:<10.3f} {pd_:<10.3f} {pa:<10.3f} {lh:<10.2f} {la:.2f}")

In [ ]:
# Evaluate Poisson model on test matches
def evaluate_poisson(test_df, attack, defense, home_adv):
    correct_result = 0
    total = 0
    errors = []

    for _, row in test_df.iterrows():
        home, away = row["home_team_name"], row["away_team_name"]
        actual_hg, actual_ag = row["home_team_score"], row["away_team_score"]

        ph, pd_, pa, lh, la = predict_match(home, away, attack, defense, home_adv)

        # Predicted result = highest probability outcome
        pred_result = ["home_win","draw","away_win"][[ph,pd_,pa].index(max(ph,pd_,pa))]
        actual_result = "home_win" if actual_hg > actual_ag else ("draw" if actual_hg == actual_ag else "away_win")

        correct_result += (pred_result == actual_result)
        errors.append(abs(actual_hg - lh) + abs(actual_ag - la))
        total += 1

    acc = correct_result / total
    mae = np.mean(errors)
    print(f"Result accuracy  : {acc:.3f}  ({correct_result}/{total} correct)")
    print(f"Goal prediction MAE : {mae:.3f} goals/team/match")
    print(f"\nBaseline (always predict home win): {(test_df['home_team_score'] > test_df['away_team_score']).mean():.3f}")
    return acc, mae

print("=== Poisson Model Evaluation on Test Set (2018 + 2022) ===")
acc, mae = evaluate_poisson(test_matches, attack, defense, home_adv)

## 5. Monte Carlo Tournament Simulator

Simulates the full World Cup bracket 10,000 times.

**How it works:**
1. Group stage: every team plays every other team in their group (round-robin), top 2 advance on points → goal difference → goals scored
2. Knockout stage: single match; draws resolved by penalty shootout (modelled as coin flip)
3. Record the winner of each simulation
4. Final probability = % of simulations won

**Key design choice:** Attack/defense ratings are retrained on all data up to (but not including)
the tournament being simulated — no leakage.


In [ ]:
def simulate_group_stage(groups, attack, defense):
    """
    groups: dict of {group_name: [team1, team2, team3, team4]}
    Returns list of teams that advance (2 per group).
    """
    qualifiers = []   # (group_winner, group_runner_up) pairs

    for group_name, teams in groups.items():
        pts = {t: 0 for t in teams}
        gd  = {t: 0 for t in teams}
        gf  = {t: 0 for t in teams}

        for i, home in enumerate(teams):
            for away in teams[i+1:]:
                lh = attack.get(home, AVG_ATTACK) * defense.get(away, AVG_DEFENSE)
                la = attack.get(away, AVG_ATTACK) * defense.get(home, AVG_DEFENSE)
                hg = np.random.poisson(lh)
                ag = np.random.poisson(la)

                gd[home] += hg - ag; gd[away] += ag - hg
                gf[home] += hg;      gf[away] += ag

                if hg > ag:   pts[home] += 3
                elif hg == ag: pts[home] += 1; pts[away] += 1
                else:         pts[away] += 3

        ranked = sorted(teams, key=lambda t: (pts[t], gd[t], gf[t]), reverse=True)
        qualifiers.append((ranked[0], ranked[1]))  # (1st, 2nd)

    return qualifiers

def simulate_knockout_match(team_a, team_b, attack, defense):
    """Single elimination match. Draws → penalty shootout (50/50)."""
    lh = attack.get(team_a, AVG_ATTACK) * defense.get(team_b, AVG_DEFENSE)
    la = attack.get(team_b, AVG_ATTACK) * defense.get(team_a, AVG_DEFENSE)
    hg = np.random.poisson(lh)
    ag = np.random.poisson(la)
    if hg > ag: return team_a
    if ag > hg: return team_b
    return team_a if np.random.random() > 0.5 else team_b

def simulate_knockout_bracket(qualifiers):
    """
    Standard 32-team WC bracket:
    R16: 1A vs 2B, 1C vs 2D, 1E vs 2F, 1G vs 2H,
         1B vs 2A, 1D vs 2C, 1F vs 2E, 1H vs 2G
    Then QF, SF, Final.
    """
    # qualifiers is list of 8 (winner, runner_up) tuples: A,B,C,D,E,F,G,H
    if len(qualifiers) < 8:
        # Handle pre-1998 formats with fewer groups (flatten and bracket)
        all_teams = [t for pair in qualifiers for t in pair]
        while len(all_teams) > 1:
            next_round = []
            for i in range(0, len(all_teams), 2):
                if i+1 < len(all_teams):
                    winner = simulate_knockout_match(all_teams[i], all_teams[i+1], attack, defense)
                    next_round.append(winner)
                else:
                    next_round.append(all_teams[i])
            all_teams = next_round
        return all_teams[0]

    A,B,C,D,E,F,G,H = qualifiers
    r16 = [
        simulate_knockout_match(A[0], B[1], attack, defense),
        simulate_knockout_match(C[0], D[1], attack, defense),
        simulate_knockout_match(E[0], F[1], attack, defense),
        simulate_knockout_match(G[0], H[1], attack, defense),
        simulate_knockout_match(B[0], A[1], attack, defense),
        simulate_knockout_match(D[0], C[1], attack, defense),
        simulate_knockout_match(F[0], E[1], attack, defense),
        simulate_knockout_match(H[0], G[1], attack, defense),
    ]
    qf = [
        simulate_knockout_match(r16[0], r16[1], attack, defense),
        simulate_knockout_match(r16[2], r16[3], attack, defense),
        simulate_knockout_match(r16[4], r16[5], attack, defense),
        simulate_knockout_match(r16[6], r16[7], attack, defense),
    ]
    sf = [
        simulate_knockout_match(qf[0], qf[1], attack, defense),
        simulate_knockout_match(qf[2], qf[3], attack, defense),
    ]
    return simulate_knockout_match(sf[0], sf[1], attack, defense)

print("✅ Simulation functions defined")

In [ ]:
# Define the 2022 World Cup groups
# (change this for 2026 prediction once the draw is known)
GROUPS_2022 = {
    "A": ["Qatar", "Ecuador", "Senegal", "Netherlands"],
    "B": ["England", "Iran", "United States", "Wales"],
    "C": ["Argentina", "Saudi Arabia", "Mexico", "Poland"],
    "D": ["France", "Australia", "Denmark", "Tunisia"],
    "E": ["Spain", "Costa Rica", "Germany", "Japan"],
    "F": ["Belgium", "Canada", "Morocco", "Croatia"],
    "G": ["Brazil", "Serbia", "Switzerland", "Cameroon"],
    "H": ["Portugal", "Ghana", "Uruguay", "South Korea"],
}

# Retrain on pre-2022 data for this prediction
attack_22, defense_22, home_adv_22 = fit_poisson_ratings(
    matches[matches["year"] < 2022]
)

# Run simulation
N_SIMS = 10_000
win_counts = defaultdict(int)

for sim in range(N_SIMS):
    qualifiers = simulate_group_stage(GROUPS_2022, attack_22, defense_22)
    winner     = simulate_knockout_bracket(qualifiers)
    win_counts[winner] += 1

# Results
win_probs = pd.DataFrame([
    {"team": team, "win_probability": count / N_SIMS}
    for team, count in win_counts.items()
]).sort_values("win_probability", ascending=False).reset_index(drop=True)

print(f"=== 2022 World Cup — Tournament Winner Probabilities ===")
print(f"(Based on {N_SIMS:,} simulations)\n")
display(win_probs.head(15).assign(
    win_probability=lambda d: d["win_probability"].map("{:.1%}".format)
))
print(f"\nActual winner: Argentina")
actual_rank = win_probs[win_probs["team"]=="Argentina"].index[0] + 1
print(f"Argentina ranked #{actual_rank} by the model")

In [ ]:
# Visualise win probabilities
fig, ax = plt.subplots(figsize=(12, 7))
top15 = win_probs.head(15).copy()
colors = ["#FFD700" if t == "Argentina" else "#4A90D9" for t in top15["team"]]

bars = ax.barh(top15["team"][::-1], top15["win_probability"][::-1], color=colors[::-1])
ax.set_xlabel("Probability of Winning Tournament")
ax.set_title(f"2022 World Cup — Win Probabilities\n({N_SIMS:,} Monte Carlo simulations | gold = actual winner)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

for bar, prob in zip(bars, top15["win_probability"][::-1]):
    ax.text(prob + 0.002, bar.get_y() + bar.get_height()/2,
            f"{prob:.1%}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 6. Baseline Comparison — Does the Model Beat Elo?

The model is only useful if it beats a naive baseline.
The simplest baseline: **always pick the team with the highest Elo rating** as the winner.


In [ ]:
def elo_baseline_predictions(team_features, test_years):
    """Pick the team with highest Elo in each test tournament."""
    results = {}
    for yr in test_years:
        yr_data = team_features[team_features["year"] == yr]
        if yr_data.empty or "elo" not in yr_data.columns:
            continue
        top_team = yr_data.loc[yr_data["elo"].idxmax(), "team_name"]
        actual   = WINNER_MAP.get(f"WC-{yr}", "Unknown")
        results[yr] = {"elo_pick": top_team, "actual_winner": actual,
                        "correct": top_team == actual}
    return pd.DataFrame(results).T

elo_results = elo_baseline_predictions(team_features, TEST_YEARS)
print("=== Elo Baseline ===")
display(elo_results)

print("\n=== Monte Carlo Model ===")
for yr, actual in [(2022, "Argentina")]:
    # The model already ran for 2022 above
    mc_pick = win_probs.iloc[0]["team"]
    mc_prob = win_probs.iloc[0]["win_probability"]
    actual_prob = win_probs[win_probs["team"]==actual]["win_probability"].values
    actual_prob = actual_prob[0] if len(actual_prob) > 0 else 0
    print(f"  {yr}: Model pick = {mc_pick} ({mc_prob:.1%}) | Actual = {actual} ({actual_prob:.1%})")

print("\n→ Key insight: a model that gives non-trivial probability to the actual winner")
print("  is more useful than a point prediction, even if the top pick is wrong.")

## 7. Award Prediction Models

**Approach:** Ranking model — score every player, rank them, top player = predicted winner.

We use Ridge Regression trained on historical award winners.
The model outputs a score per player; the player with the highest score in a tournament is the prediction.

**Important caveat on Best Player and Young Player:**
These are jury votes influenced by narrative, not just statistics.
The model can only predict *who had the best measurable tournament* — which correlates with
the award but doesn't fully explain it. Expect lower accuracy here than Golden Boot.


In [ ]:
FEAT_COLS_PLAYER = ["goals_scored","penalty_goals","open_play_goals",
                     "matches_played","matches_started","wc_exp"]

def prepare_player_data(df, feature_cols):
    df = df.copy()
    for col in feature_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    return df

train_pl = prepare_player_data(train_players, FEAT_COLS_PLAYER)
test_pl  = prepare_player_data(test_players,  FEAT_COLS_PLAYER)

scaler = StandardScaler()
X_train = scaler.fit_transform(train_pl[FEAT_COLS_PLAYER])
X_test  = scaler.transform(test_pl[FEAT_COLS_PLAYER])

print(f"Train players: {len(train_pl):,}")
print(f"Test players : {len(test_pl):,}")
print(f"Features     : {FEAT_COLS_PLAYER}")

In [ ]:
def train_award_model(train_df, X_train, target_col, alpha=1.0):
    """Train Ridge regression where 1 = award winner, 0 = everyone else."""
    y = train_df[target_col].astype(int).values
    model = Ridge(alpha=alpha)
    model.fit(X_train, y)
    return model

def predict_award(model, test_df, X_test, target_col, top_n=5):
    """Score all players in each test tournament and rank them."""
    scores = model.predict(X_test)
    df = test_df.copy()
    df["award_score"] = scores

    results = []
    for yr in sorted(df["year"].unique()):
        yr_df = df[df["year"] == yr].copy()
        yr_df = yr_df.sort_values("award_score", ascending=False).reset_index(drop=True)
        yr_df["predicted_rank"] = yr_df.index + 1

        actual_winner = yr_df[yr_df[target_col] == True]
        if len(actual_winner) > 0:
            actual_rank = actual_winner.iloc[0]["predicted_rank"]
            actual_name = actual_winner.iloc[0].get("player_name","Unknown")
        else:
            actual_rank = None
            actual_name = "Unknown"

        results.append({
            "year":          yr,
            "model_pick":    yr_df.iloc[0].get("player_name","Unknown"),
            "model_pick_team": yr_df.iloc[0].get("team_name","Unknown"),
            "actual_winner": actual_name,
            "actual_rank":   actual_rank,
            "correct":       actual_rank == 1,
        })

        print(f"  {yr} | Model: {yr_df.iloc[0].get('player_name','?'):<25} | Actual: {actual_name}")

    return pd.DataFrame(results)

# ── Golden Boot ──────────────────────────────────────────────────────────────
print("\n=== Golden Boot Model ===")
gb_model   = train_award_model(train_pl, X_train, "won_golden_boot")
gb_results = predict_award(gb_model, test_pl, X_test, "won_golden_boot")

print(f"\nGolden Boot accuracy: {gb_results['correct'].mean():.0%} ({gb_results['correct'].sum()}/{len(gb_results)})")
print("→ Note: shared Golden Boots (same goals) counted as 1 winner in the dataset")

In [ ]:
# ── Best Player ───────────────────────────────────────────────────────────────
print("=== Best Player Model (Golden Ball) ===")
bp_model   = train_award_model(train_pl, X_train, "won_best_player")
bp_results = predict_award(bp_model, test_pl, X_test, "won_best_player")
print(f"\nBest Player accuracy: {bp_results['correct'].mean():.0%}")
print("→ Expected to be lower — jury vote not purely stat-driven")

# ── Young Player ──────────────────────────────────────────────────────────────
print("\n=== Young Player Model (U21 at tournament start) ===")
# Filter to U23 players only (eligibility threshold varies; 23 is safe)
train_young = train_pl[train_pl["age"] <= 23] if "age" in train_pl.columns else train_pl
test_young  = test_pl[test_pl["age"] <= 23]   if "age" in test_pl.columns else test_pl

if len(train_young) > 0:
    X_train_y = scaler.fit_transform(train_young[FEAT_COLS_PLAYER])
    X_test_y  = scaler.transform(test_young[FEAT_COLS_PLAYER])
    yp_model   = train_award_model(train_young, X_train_y, "won_young_player")
    yp_results = predict_award(yp_model, test_young, X_test_y, "won_young_player")
    print(f"\nYoung Player accuracy: {yp_results['correct'].mean():.0%}")

## 8. Feature Importance — What Drives Each Award?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Award Model — Feature Importance (Ridge Coefficients)", fontsize=13)

for ax, (model, title) in zip(axes, [
    (gb_model, "Golden Boot"),
    (bp_model, "Best Player"),
    (yp_model if len(train_young) > 0 else gb_model, "Young Player"),
]):
    coefs = pd.Series(model.coef_, index=FEAT_COLS_PLAYER).sort_values()
    colors = ["#E74C3C" if c < 0 else "#2ECC71" for c in coefs]
    ax.barh(coefs.index, coefs.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Coefficient")

plt.tight_layout()
plt.show()
print("\nGreen = positive contribution | Red = negative contribution")

## 9. Prediction Summary — 2022 World Cup

In [ ]:
print("=" * 55)
print("  2022 WORLD CUP — MODEL PREDICTIONS vs ACTUALS")
print("=" * 55)

# Tournament winner
mc_pick  = win_probs.iloc[0]["team"]
mc_prob  = win_probs.iloc[0]["win_probability"]
arg_prob = win_probs[win_probs["team"]=="Argentina"]["win_probability"].values
arg_prob = arg_prob[0] if len(arg_prob) > 0 else 0.0

print(f"\n🏆 Tournament Winner")
print(f"   Model pick   : {mc_pick} ({mc_prob:.1%})")
print(f"   Actual winner: Argentina ({arg_prob:.1%} assigned by model)")

# Golden Boot
yr2022_gb = gb_results[gb_results["year"]==2022]
if len(yr2022_gb):
    row = yr2022_gb.iloc[0]
    print(f"\n👟 Golden Boot")
    print(f"   Model pick   : {row['model_pick']} ({row['model_pick_team']})")
    print(f"   Actual winner: {row['actual_winner']}")
    print(f"   Actual rank  : #{int(row['actual_rank']) if row['actual_rank'] else '?'}")

# Best Player
yr2022_bp = bp_results[bp_results["year"]==2022]
if len(yr2022_bp):
    row = yr2022_bp.iloc[0]
    print(f"\n⭐ Best Player")
    print(f"   Model pick   : {row['model_pick']} ({row['model_pick_team']})")
    print(f"   Actual winner: {row['actual_winner']}")
    print(f"   Actual rank  : #{int(row['actual_rank']) if row['actual_rank'] else '?'}")

print("=" * 55)
print("\n→ Next step: replace Poisson + Ridge baseline with TensorFlow DNN")

## 10. Save Model Outputs

In [ ]:
from google.colab import files as colab_files

# Save win probabilities for the dashboard/Streamlit app
win_probs.to_csv("win_probabilities_2022.csv", index=False)

# Save award predictions
gb_results.to_csv("golden_boot_predictions.csv", index=False)
bp_results.to_csv("best_player_predictions.csv", index=False)

for fname in ["win_probabilities_2022.csv",
              "golden_boot_predictions.csv",
              "best_player_predictions.csv"]:
    colab_files.download(fname)
    print(f"✅ {fname}")

## ✅ Done — Baseline Model Complete

### What you built

| Component | Method | Status |
|---|---|---|
| Match prediction | Poisson regression | ✅ Baseline |
| Tournament simulation | Monte Carlo 10,000 runs | ✅ Baseline |
| Golden Boot | Ridge regression ranker | ✅ Baseline |
| Best Player | Ridge regression ranker | ✅ Baseline |
| Young Player | Ridge regression ranker (U23) | ✅ Baseline |

### Honest assessment of the baseline

The Poisson model captures team strength but ignores:
- Form going into the tournament (Elo, recent results)
- Squad quality beyond historical goal scoring
- Matchup-specific dynamics

The Ridge rankers for awards are heavily biased toward goals scored —
which works for Golden Boot but is a poor proxy for Best Player and Young Player.

### Next notebook — TensorFlow DNN Upgrade
Replace the Poisson + Ridge baseline with:
1. **DNN match outcome model** — takes Elo, form, squad features as input
2. **DNN award scorer** — takes full player feature vector, outputs award probability
3. Compare DNN vs baseline on the same test years
4. Use the better model for the final 2026 predictions
